In [1]:
# Standard library imports
import os
import random
import sys

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cProfile
import pstats

# Configure matplotlib
plt.style.use('fig.style')
figsize = (8,4)

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
from src.seq_embedder import SeqEmbedder
def test_gauge_fixing(theta_series, theta_fixed_series, L, alphabet, num_seqs=100, embedder=None):
    features = list(theta_series.index)
    
    # Check that gauge matches for every sequence
    if embedder is None:
        embedder = SeqEmbedder(features=features, L=L)

    # Check that gauge matches for every sequence
    for _ in range(num_seqs):
        
        # Choose a random sequence
        seq = ''.join(np.random.choice(alphabet, size=L))
        
        # Embed the sequence
        x = embedder.embed(seq)
        
        f = x@theta_series
        f_fixed = x@theta_fixed_series
        
        # Check that the two function values are close      
        assert np.isclose(f, f_fixed), f'{f=}\n{f_fixed=}'
        
    print(f'Tested {num_seqs} random sequences; all passed.')

In [3]:
from src import get_alphabet
L = 55
alphabet = get_alphabet('protein')
alpha = len(alphabet)
bg_df = pd.DataFrame(index=range(L), columns=alphabet, data=1.0/alpha)

# Create theta_series
from src import get_pairwise_model_features
features = get_pairwise_model_features(L=L, alphabet=alphabet)
embedder = SeqEmbedder(features=features, L=L)
values = np.random.normal(size=len(features))
theta_series = pd.Series(data=values, index=features)
theta_series


((), )            1.206148
((0,), A)        -0.812553
((0,), C)         0.051641
((0,), D)        -0.839120
((0,), E)        -1.732445
                    ...   
((53, 54), YS)    1.586696
((53, 54), YT)    0.148176
((53, 54), YV)    0.903875
((53, 54), YW)    1.122323
((53, 54), YY)   -2.182763
Length: 595101, dtype: float64

In [8]:
from src.fix_gauge_pairwise import fix_gauge_pairwise_theta_series

# Fix gauge of theta_series
with cProfile.Profile() as profiler:
    theta_fixed_series = fix_gauge_pairwise_theta_series(theta_series, p_lc=bg_df.values, alphabet=alphabet, L=L)
stats = pstats.Stats(profiler).strip_dirs().sort_stats('cumulative')
stats.print_stats(10)

# Test gauge fixing
test_gauge_fixing(theta_series=theta_series, theta_fixed_series=theta_fixed_series, L=L, alphabet=alphabet, num_seqs=10, embedder=embedder)

         2546652 function calls (2546540 primitive calls) in 9.515 seconds

   Ordered by: cumulative time
   List reduced from 325 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.006    0.006    5.714    5.714 fix_gauge_pairwise.py:62(fix_gauge_pairwise_theta_series)
        1    5.655    5.655    5.708    5.708 pairwise_theta_dict_to_series.py:5(pairwise_theta_dict_to_series)
        1    0.000    0.000    0.384    0.384 __init__.py:38(get_pairwise_model_features)
        1    0.007    0.007    0.384    0.384 get_features_upto_order.py:8(get_features_upto_order)
        4    0.000    0.000    0.348    0.087 sort_features.py:5(sort_features)
        6    0.139    0.023    0.348    0.058 {built-in method builtins.sorted}
  1190202    0.151    0.000    0.209    0.000 sort_features.py:6(<lambda>)
        4    0.000    0.000    0.206    0.052 get_features_upto_order.py:25(<genexpr>)
        3    0.029    0.010    0.206 

In [9]:
# Convert to dict
from src.pairwise_theta_series_to_dict import pairwise_theta_series_to_dict
theta_dict = pairwise_theta_series_to_dict(theta_series, alphabet=alphabet, L=L)

# Fix gauge of theta_series
from src.fix_gauge_pairwise import fix_gauge_pairwise_theta_dict
with cProfile.Profile() as profiler:
    theta_fixed_dict = fix_gauge_pairwise_theta_dict(theta_dict, p_lc=bg_df.values)
stats = pstats.Stats(profiler).strip_dirs().sort_stats('cumulative')
stats.print_stats(10)

# Test gauge fixing
test_gauge_fixing(theta_series=theta_series, theta_fixed_series=theta_fixed_series, L=L, alphabet=alphabet, num_seqs=10, embedder=embedder)

         71 function calls in 0.020 seconds

   Ordered by: cumulative time
   List reduced from 16 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.017    0.017    0.020    0.020 fix_gauge_pairwise.py:8(fix_gauge_pairwise_theta_dict)
       10    0.000    0.000    0.003    0.000 fromnumeric.py:2349(sum)
       10    0.000    0.000    0.003    0.000 fromnumeric.py:69(_wrapreduction)
       10    0.003    0.000    0.003    0.000 {method 'reduce' of 'numpy.ufunc' objects}
        1    0.000    0.000    0.000    0.000 cProfile.py:119(__exit__)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.000    0.000    0.000    0.000 frame.py:12590(values)
        1    0.000    0.000    0.000    0.000 managers.py:1633(as_array)
       10    0.000    0.000    0.000    0.000 {built-in method builtins.isinstance}
       10    0.000    0.000    0.000    0.000 {method 'items' o

In [10]:
with cProfile.Profile() as profiler:
    test_gauge_fixing(theta_series=theta_series, theta_fixed_series=theta_fixed_series, L=L, alphabet=alphabet, num_seqs=10, embedder=embedder)
stats = pstats.Stats(profiler).strip_dirs().sort_stats('cumulative')
stats.print_stats(10)

Tested 10 random sequences; all passed.
         5953002 function calls (5952983 primitive calls) in 1.979 seconds

   Ordered by: cumulative time
   List reduced from 227 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      7/5    0.006    0.001    3.896    0.779 selectors.py:558(select)
      8/7    0.002    0.000    1.991    0.284 base_events.py:1915(_run_once)
       10    0.676    0.068    1.623    0.162 seq_embedder.py:27(embed)
  5951010    0.957    0.000    0.957    0.000 {method 'match' of 're.Pattern' objects}
       10    0.166    0.017    0.166    0.017 {built-in method numpy.array}
        1    0.001    0.001    0.038    0.038 2345477463.py:2(test_gauge_fixing)
      8/5    0.000    0.000    0.020    0.004 events.py:86(_run)
        1    0.000    0.000    0.020    0.020 decorator.py:232(fun)
        1    0.000    0.000    0.020    0.020 history.py:90(only_when_enabled)
      7/5    0.000    0.000    0.020    0.004 {me